# Step 8 - RAGAS Evaluation (RQ2 Framework A answers)

**Owner:** Raj · **Depends on:** Step 7a (Yan-Bo, Framework A generation, done 2026-07-12)

**Scope.** Per `rq2_plan.md` Decision 3 and `rq2_methodology.md`, RAGAS was
deliberately kept OUT of RQ2 and attached here instead. NDCG@10, MRR, and SPD
are already measured (Step 6 re-ranking / RQ3, Step 5b fairness audit / RQ1);
this notebook does NOT recompute them. It adds the one new thing Step 8 owns:
RAG **answer-quality** scores (Faithfulness, Answer Relevancy, Context
Precision) for the 100 neutral-query answers Yan-Bo already generated, then
consolidates all four families of metrics into one file for Step 9.

**Inputs (attach as Kaggle datasets):**
- `yanbochen928/step7-frameworka-for-raj` -> `rq2_gen_checkpoint.jsonl`
  (100 generated answers + citations, from Step 7a)
- `yanbochen928/fairsearch-qbio-processed-raj-jici-yb` -> `qbio_papers.json`
  (55,300-paper corpus; needed to turn `retrieved_paper_ids` into the actual
  abstract text RAGAS needs as "contexts" - the checkpoint only has IDs)

**Reference-free, no ground truth.** Faithfulness, Answer Relevancy, and
Context Precision (without-reference variant) only need question + answer +
retrieved contexts. We do not have or need human-written reference answers.

**Judge model.** `gemini-3.1-flash-lite`, temperature 0 - same model and same
free-tier constraints as Step 7a (see `rq2_plan.md` Decision 4 for the full
deprecation saga). Context Precision also needs an embedding model; we reuse
`all-MiniLM-L6-v2` (already used project-wide for retrieval) instead of a
Gemini embedding call, to avoid burning extra quota on a model that is not
the thing being evaluated.

**Outputs:**
- `/kaggle/working/rq2_frameworkA_ragas_checkpoint.jsonl` (per-query, resumable)
- `/kaggle/working/rq2_frameworkA_ragas_results.json` (aggregated + bootstrap CI)
- `/kaggle/working/step8_consolidated_summary.json` (RAGAS + existing NDCG@10 /
  MRR / SPD numbers, one file for the Step 9 dashboard to read)


In [1]:
import os, time
from kaggle_secrets import UserSecretsClient
from langchain_google_genai import ChatGoogleGenerativeAI

# reload key (in case of kernel restart)
os.environ["GOOGLE_API_KEY"] = UserSecretsClient().get_secret("GEMINI_API_KEY").strip()

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0)

# call 1: basic sanity
start = time.time()
r = llm.invoke("Reply with exactly the word: OK")
print(f"call 1 ({time.time()-start:.1f}s): {r.content}")

# call 2: same, immediately after (catches per-minute limits)
start = time.time()
r = llm.invoke("Reply with exactly the word: HELLO")
print(f"call 2 ({time.time()-start:.1f}s): {r.content}")

print("\nKey works. If call 2 was fast (<2s), quota headroom is good — proceed.")
print("If either call 429s, paste the full error including quotaId.")

ModuleNotFoundError: No module named 'langchain_google_genai'

## Cell 0 - Environment check: install RAGAS + LangChain Gemini + embeddings

Goal: get a working RAGAS judge stack without guessing package names. We need:
- `ragas` (the evaluation library itself)
- `langchain-google-genai` (so the Gemini judge model can be wrapped for RAGAS
  via `LangchainLLMWrapper`)
- `langchain-huggingface` + `sentence-transformers` (local embedding model for
  Context Precision, so we are not spending Gemini quota on embeddings)

Internet must be ON (Settings -> Internet -> On) for the installs to work.
This cell only installs/reports versions; no API calls yet.


In [2]:
# Cell 0: check + install RAGAS stack, with compatibility stub
import importlib, subprocess, sys, types

def pkg_version(mod_name):
    try:
        m = importlib.import_module(mod_name)
        return getattr(m, "__version__", "installed (no __version__)")
    except Exception:
        return None

needed = {
    "ragas": "ragas==0.3.9",
    "langchain_google_genai": "langchain-google-genai",
    "langchain_huggingface": "langchain-huggingface",
    "sentence_transformers": "sentence-transformers",
}

print("BEFORE install:")
missing_pip_names = []
for mod, pip_name in needed.items():
    v = pkg_version(mod)
    print(f"  {mod:26s} -> {v}")
    if v is None:
        missing_pip_names.append(pip_name)

if missing_pip_names:
    print("\ninstalling:", missing_pip_names)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing_pip_names, check=True)
    importlib.invalidate_caches()
else:
    print("\nall present -> no install needed")

# --- compatibility stub (ragas 0.3.9 still imports an old langchain_community
# path this environment's langchain-community no longer has; we don't use
# Vertex AI at all, so we stub it out rather than fight the dependency tree) ---
if "langchain_community.chat_models.vertexai" not in sys.modules:
    stub = types.ModuleType("langchain_community.chat_models.vertexai")
    stub.ChatVertexAI = object
    sys.modules["langchain_community.chat_models.vertexai"] = stub

import langchain_community.llms as _llms_mod
if not hasattr(_llms_mod, "VertexAI"):
    _llms_mod.VertexAI = object

print("\nAFTER install:")
for mod in needed:
    print(f"  {mod:26s} -> {pkg_version(mod)}")

assert pkg_version("ragas") is not None, "ragas still not importable"

BEFORE install:
  ragas                      -> None
  langchain_google_genai     -> None
  langchain_huggingface      -> None
  sentence_transformers      -> 5.4.1

installing: ['ragas==0.3.9', 'langchain-google-genai', 'langchain-huggingface']
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.7/366.7 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
/tmp/ipykernel_58/2136843702.py:41: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward s


AFTER install:
  ragas                      -> 0.3.9
  langchain_google_genai     -> 4.3.2
  langchain_huggingface      -> installed (no __version__)
  sentence_transformers      -> 5.4.1


## Cell 1 - Load data and build a paper lookup table

Same auto-detect-by-filename approach as the Step 7 notebooks (Kaggle input
paths differ from local paths, so we `rglob` under `/kaggle/input` instead of
hardcoding a path).

Two files (from the two mounted datasets):
- `rq2_gen_checkpoint.jsonl` : 100 generated answers, one JSON object per line
  (query_id, query_text, answer_text, retrieved_paper_ids, cited_pids_unique, ...)
- `qbio_papers.json` : corpus metadata (title + abstract), used to turn
  `retrieved_paper_ids` into actual text for RAGAS "contexts"

We hard-assert we got 100 records (Framework A ran on the 100 neutral
queries only) before continuing.


In [3]:
# Cell 1: load data, auto-detect paths, build paper lookup
import json
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

def find_file(*candidates):
    """Find a file by exact name (trying each candidate in order) anywhere under /kaggle/input."""
    for filename in candidates:
        hits = list(INPUT_ROOT.rglob(filename))
        if hits:
            if len(hits) > 1:
                print(f"WARNING: multiple copies of {filename}: {hits}. Using first.")
            return hits[0]
    raise AssertionError(f"NOT FOUND: none of {candidates} under {INPUT_ROOT}")

# 1) locate files (Kaggle sometimes appends .txt to .jsonl on upload)
path_ckpt   = find_file("rq2_gen_checkpoint.jsonl", "rq2_gen_checkpoint.jsonl.txt")
path_papers = find_file("qbio_papers.json")
print("Found:")
print("  rq2_gen_checkpoint ->", path_ckpt)
print("  qbio_papers.json   ->", path_papers)

# 2) load checkpoint (one JSON object per line, tolerant of a corrupt last line)
gen_records = []
with open(path_ckpt) as f:
    for i, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue
        try:
            gen_records.append(json.loads(line))
        except Exception as e:
            print(f"  WARNING: skipping bad checkpoint line {i}: {str(e)[:60]}")

print(f"\nloaded {len(gen_records)} generated-answer records")
assert len(gen_records) == 100, f"expected 100 Framework A records, got {len(gen_records)}"

# 3) load corpus and build paper_id -> record lookup
papers = json.loads(path_papers.read_text())
pmap = {p["paper_id"]: p for p in papers} if isinstance(papers, list) else papers
print(f"corpus loaded: {len(pmap)} papers")

Found:
  rq2_gen_checkpoint -> /kaggle/input/datasets/yanbochen928/step7-frameworka-for-raj/rq2_gen_checkpoint.jsonl.txt
  qbio_papers.json   -> /kaggle/input/datasets/yanbochen928/fairsearch-qbio-processed-raj-jici-yb/qbio_papers.json

loaded 100 generated-answer records
corpus loaded: 55300 papers


## Cell 2 - Build the RAGAS evaluation rows

For each generated-answer record, assemble the three fields RAGAS needs:
- `user_input`  = the original query text
- `response`    = the model's generated answer (`answer_text`)
- `retrieved_contexts` = the 10 retrieved papers' title+abstract text, looked
  up from the corpus via `retrieved_paper_ids` (same 10 papers the model saw
  in Step 7a - we are scoring against exactly what it was given, not a fresh
  retrieval)

Missing abstracts get a placeholder (mirrors `build_context()` in the Step 7
notebook) so a lookup miss does not silently shrink the context list.


In [4]:
# Cell 2: build eval rows (pure data prep, no API calls)
def paper_text(pid):
    p = pmap.get(pid)
    if p is None:
        return "(title unavailable) (abstract unavailable)"
    title = (p.get("title") or "(no title)").strip()
    abstract = (p.get("abstract") or "(no abstract)").strip()
    return f"{title}\n{abstract}"

eval_rows = []
for rec in gen_records:
    eval_rows.append({
        "query_id": rec["query_id"],
        "user_input": rec["query_text"],
        "response": rec["answer_text"],
        "retrieved_contexts": [paper_text(pid) for pid in rec["retrieved_paper_ids"]],
    })

print(f"built {len(eval_rows)} eval rows")
print("\nexample row (query_id, user_input, first context truncated):")
ex = eval_rows[0]
print(" query_id:", ex["query_id"])
print(" user_input:", ex["user_input"])
print(" contexts[0][:150]:", ex["retrieved_contexts"][0][:150])


built 100 eval rows

example row (query_id, user_input, first context truncated):
 query_id: q001
 user_input: How can stochastic differential equations model gene expression noise in single cells?
 contexts[0][:150]: Stochastic Modeling and Statistical Inference of Intrinsic Noise in Gene Regulation System via Chemical Master Equation
Intrinsic noise, the stochasti


## Cell 3 - Judge LLM + embeddings setup

Same key-handling convention as Step 7: read from Kaggle Secrets, never
hardcode. Judge model is `gemini-3.1-flash-lite` at temperature 0, matching
Step 7a exactly (same model, same quota tier - see `rq2_plan.md` Decision 4
for why this specific model was chosen after several others were unavailable).

Embeddings for Context Precision use `all-MiniLM-L6-v2` locally (CPU, free,
no quota) rather than a Gemini embedding call - Context Precision only needs
embeddings to judge relevance ranking, not the model actually under test.


In [5]:
# Cell 3: judge LLM + embeddings, wrapped for RAGAS
import os
from kaggle_secrets import UserSecretsClient
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# 1) read the key from Kaggle Secrets (never hardcode the key itself)
GEMINI_API_KEY = UserSecretsClient().get_secret("GEMINI_API_KEY")
GEMINI_API_KEY = GEMINI_API_KEY.strip() if GEMINI_API_KEY else GEMINI_API_KEY
assert GEMINI_API_KEY, "key not found; check Secrets label = GEMINI_API_KEY and that it is attached"
os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY
print("key loaded from Secrets: OK (length =", len(GEMINI_API_KEY), ")")

JUDGE_MODEL = "gemini-3.1-flash-lite"
JUDGE_TEMPERATURE = 0

evaluator_llm = LangchainLLMWrapper(ChatGoogleGenerativeAI(
    model=JUDGE_MODEL,
    temperature=JUDGE_TEMPERATURE,
))

evaluator_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
)

print(f"judge model: {JUDGE_MODEL} | temperature: {JUDGE_TEMPERATURE}")
print("embeddings: all-MiniLM-L6-v2 (local, CPU)")


key loaded from Secrets: OK (length = 53 )


/tmp/ipykernel_58/263946779.py:19: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(ChatGoogleGenerativeAI(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

judge model: gemini-3.1-flash-lite | temperature: 0
embeddings: all-MiniLM-L6-v2 (local, CPU)


/tmp/ipykernel_58/263946779.py:24: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = LangchainEmbeddingsWrapper(


## Cell 4 - Define the three RAGAS metrics (reference-free)

- `Faithfulness` - does the answer stick to what the retrieved papers say?
- `ResponseRelevancy` - does the answer actually address the question asked?
- `LLMContextPrecisionWithoutReference` - are the useful retrieved papers
  ranked near the top? (the without-reference variant needs only
  `user_input`, `response`, `retrieved_contexts` - no ground truth)

All three take the same `llm=` / `embeddings=` we just built.


In [6]:
# Cell 4: instantiate metrics, split into cheap (run now) vs expensive (hold)
from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextPrecisionWithoutReference

faithfulness = Faithfulness(llm=evaluator_llm)
answer_relevancy = ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings)
context_precision = LLMContextPrecisionWithoutReference(llm=evaluator_llm)

# Per Yan-Bo's plan: cheap first, hold expensive until quota strategy confirmed.
# Faithfulness = ~3-4 judge calls/query (decompose answer -> claims, verify each)
# ResponseRelevancy = ~1 call/query (generate reverse-question, cosine-sim to original)
# LLMContextPrecisionWithoutReference = 10 calls/query (one per retrieved context)
METRICS_CHEAP = {
    "faithfulness": faithfulness,
    "answer_relevancy": answer_relevancy,
}
METRICS_EXPENSIVE = {
    "context_precision": context_precision,
}
print("cheap metrics (run now):", list(METRICS_CHEAP.keys()))
print("expensive metrics (hold):", list(METRICS_EXPENSIVE.keys()))

cheap metrics (run now): ['faithfulness', 'answer_relevancy']
expensive metrics (hold): ['context_precision']


In [7]:
# Override answer_relevancy with strictness=1
# gemini-3.1-flash-lite doesn't support multiple candidates per request.
# strictness=1 asks for a single reverse-question instead of the default 3.
from ragas.metrics import ResponseRelevancy

answer_relevancy = ResponseRelevancy(
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    strictness=1,
)

METRICS_CHEAP = {
    "faithfulness": faithfulness,
    "answer_relevancy": answer_relevancy,
}
print("answer_relevancy strictness set to 1")
print("METRICS_CHEAP:", list(METRICS_CHEAP.keys()))

answer_relevancy strictness set to 1
METRICS_CHEAP: ['faithfulness', 'answer_relevancy']


## Cell 5 - Per-query scoring with checkpoint + resume

Same safety pattern as Step 7a's batch generation, because we are on the same
free-tier judge model and can hit the same quota walls:
- per-query checkpoint: after each query's 3 scores are computed, append one
  line to `/kaggle/working/rq2_frameworkA_ragas_checkpoint.jsonl`
- resume: on (re)run, skip query_ids already in the checkpoint
- per-query try/except: one failed query is logged and skipped, not fatal

Each metric's `single_turn_ascore` is async, so we run them with `asyncio`.


In [8]:
from ragas import SingleTurnSample

sample = SingleTurnSample(
    user_input=eval_rows[0]["user_input"],
    response=eval_rows[0]["response"],
    retrieved_contexts=eval_rows[0]["retrieved_contexts"],
)

f_score = await faithfulness.single_turn_ascore(sample)
ar_score = await answer_relevancy.single_turn_ascore(sample)
print(f"q001 faithfulness: {f_score}")
print(f"q001 answer_relevancy: {ar_score}")

q001 faithfulness: 1.0
q001 answer_relevancy: 0.9085660932426163


In [9]:
# Cell 5: per-query RAGAS scoring for CHEAP metrics only, checkpointed
import asyncio, time, json, os
from ragas import SingleTurnSample

CKPT_PATH = "/kaggle/working/rq2_frameworkA_ragas_cheap_checkpoint.jsonl"
METRICS_RUN = METRICS_CHEAP
SLEEP = 8            # CHANGED: was 3, bump to give flash-lite RPM more room
MAX_RETRIES = 5

# 1) load checkpoint (resume support)
done_ids = set()
if os.path.exists(CKPT_PATH):
    with open(CKPT_PATH) as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                done_ids.add(json.loads(line)["query_id"])
            except Exception as e:
                print(f"  WARNING: skipping bad checkpoint line {i}: {str(e)[:60]}")
print(f"resume: {len(done_ids)} queries already scored (cheap metrics)")

todo = [r for r in eval_rows if r["query_id"] not in done_ids]
print(f"to score now: {len(todo)} queries")
print(f"metrics this run: {list(METRICS_RUN.keys())}\n")

async def score_row(row):
    sample = SingleTurnSample(
        user_input=row["user_input"],
        response=row["response"],
        retrieved_contexts=row["retrieved_contexts"],
    )
    scores = {}
    for name, metric in METRICS_RUN.items():
        scores[name] = await metric.single_turn_ascore(sample)
    return scores

async def score_with_retry(row):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return await score_row(row)
        except Exception as e:
            msg = str(e)
            # CHANGED: also treat 400s that mention rate/quota as transient,
            # not just 429/503 — flash-lite sometimes returns 400 for RPM overflow
            transient = ("429" in msg) or ("503" in msg) \
                or ("RESOURCE_EXHAUSTED" in msg) or ("UNAVAILABLE" in msg) \
                or ("400" in msg and any(k in msg.lower() for k in ["rate", "quota", "exceed", "limit"]))
            if transient and attempt < MAX_RETRIES:
                wait = 30 * attempt   # CHANGED: 30s * attempt (was 20)
                print(f"    {row['query_id']} transient error (attempt {attempt}); waiting {wait}s")
                print(f"    error preview: {msg[:200]}")
                await asyncio.sleep(wait)
                continue
            raise

async def run_batch():
    failed = []
    for idx, row in enumerate(todo, start=1):
        qid = row["query_id"]
        try:
            scores = await score_with_retry(row)
            rec = {"query_id": qid, **scores}
            with open(CKPT_PATH, "a") as f:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        except Exception as e:
            failed.append({"query_id": qid, "error": str(e)[:300]})
            print(f"  {qid} FAILED (gave up): {str(e)[:250]}")   # CHANGED: was 120, show more
        if idx % 10 == 0:
            print(f"  progress: {idx}/{len(todo)} attempted, failed so far={len(failed)}")
        time.sleep(SLEEP)
    return failed

failed = await run_batch()

# completeness check
done_ids = set()
if os.path.exists(CKPT_PATH):
    with open(CKPT_PATH) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    done_ids.add(json.loads(line)["query_id"])
                except Exception:
                    pass
target_ids = {r["query_id"] for r in eval_rows}
missing = target_ids - done_ids
print(f"\nDone. checkpoint has {len(done_ids)}/{len(target_ids)} target queries scored on cheap metrics.")
if missing:
    print(f"MISSING {len(missing)}: {sorted(missing)[:20]}")
    print("-> re-run this cell to fill the missing ones (checkpoint will resume)")
else:
    print("All target queries scored on cheap metrics. Ready for aggregation (Cell 6).")

if failed:
    print(f"\n{len(failed)} query failures logged — first few, full text:")
    for f in failed[:3]:
        print(" ", f)

resume: 0 queries already scored (cheap metrics)
to score now: 100 queries
metrics this run: ['faithfulness', 'answer_relevancy']



CancelledError: 

In [12]:
# Cell 5b: Context Precision, separate checkpoint from cheap metrics
import asyncio, time, json, os
from ragas import SingleTurnSample

CKPT_PATH_CP = "/kaggle/working/rq2_frameworkA_ragas_cp_checkpoint.jsonl"
METRICS_RUN = METRICS_EXPENSIVE   # just context_precision
SLEEP = 8
MAX_RETRIES = 5

done_ids = set()
if os.path.exists(CKPT_PATH_CP):
    with open(CKPT_PATH_CP) as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                done_ids.add(json.loads(line)["query_id"])
            except Exception as e:
                print(f"  WARNING: skipping bad checkpoint line {i}: {str(e)[:60]}")
print(f"resume: {len(done_ids)} queries already scored (context precision)")

todo = [r for r in eval_rows if r["query_id"] not in done_ids]
print(f"to score now: {len(todo)} queries")
print(f"metrics this run: {list(METRICS_RUN.keys())}")
print(f"est. API calls: ~{len(todo) * 10} (10 per query for context precision)\n")

async def score_row_cp(row):
    sample = SingleTurnSample(
        user_input=row["user_input"],
        response=row["response"],
        retrieved_contexts=row["retrieved_contexts"],
    )
    scores = {}
    for name, metric in METRICS_RUN.items():
        scores[name] = await metric.single_turn_ascore(sample)
    return scores

async def score_with_retry_cp(row):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return await score_row_cp(row)
        except Exception as e:
            msg = str(e)
            transient = ("429" in msg) or ("503" in msg) \
                or ("RESOURCE_EXHAUSTED" in msg) or ("UNAVAILABLE" in msg) \
                or ("400" in msg and any(k in msg.lower() for k in ["rate", "quota", "exceed", "limit"]))
            if transient and attempt < MAX_RETRIES:
                wait = 30 * attempt
                print(f"    {row['query_id']} transient error (attempt {attempt}); waiting {wait}s")
                print(f"    error preview: {msg[:200]}")
                await asyncio.sleep(wait)
                continue
            raise

async def run_batch_cp():
    failed = []
    for idx, row in enumerate(todo, start=1):
        qid = row["query_id"]
        try:
            scores = await score_with_retry_cp(row)
            rec = {"query_id": qid, **scores}
            with open(CKPT_PATH_CP, "a") as f:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        except Exception as e:
            failed.append({"query_id": qid, "error": str(e)[:300]})
            print(f"  {qid} FAILED (gave up): {str(e)[:250]}")
        if idx % 10 == 0:
            print(f"  progress: {idx}/{len(todo)} attempted, failed so far={len(failed)}")
        time.sleep(SLEEP)
    return failed

failed = await run_batch_cp()

done_ids = set()
if os.path.exists(CKPT_PATH_CP):
    with open(CKPT_PATH_CP) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    done_ids.add(json.loads(line)["query_id"])
                except Exception:
                    pass
target_ids = {r["query_id"] for r in eval_rows}
missing = target_ids - done_ids
print(f"\nDone. checkpoint has {len(done_ids)}/{len(target_ids)} target queries scored on context precision.")
if missing:
    print(f"MISSING {len(missing)}: {sorted(missing)[:20]}")
    print("-> re-run this cell to fill the missing ones (checkpoint will resume)")
else:
    print("All target queries scored on context precision. Ready for final aggregation.")

if failed:
    print(f"\n{len(failed)} query failures logged — first few:")
    for f in failed[:3]:
        print(" ", f)

resume: 47 queries already scored (context precision)
to score now: 53 queries
metrics this run: ['context_precision']
est. API calls: ~530 (10 per query for context precision)

    q048 transient error (attempt 1); waiting 30s
    error preview: Error calling model 'gemini-3.1-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing detail
    q048 transient error (attempt 2); waiting 60s
    error preview: Error calling model 'gemini-3.1-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing detail
    q048 transient error (attempt 3); waiting 90s
    error preview: Error calling model 'gemini-3.1-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing detail
    q048 tra

CancelledError: 

In [17]:
# Copy checkpoints to a location that shows in the output panel + print full paths
import shutil, os
src_files = [
    "/kaggle/working/rq2_frameworkA_ragas_cheap_checkpoint.jsonl",
    "/kaggle/working/rq2_frameworkA_ragas_cp_checkpoint.jsonl",
]
for src in src_files:
    if os.path.exists(src):
        print(f"OK: {src}  ({os.path.getsize(src)} bytes)")
    else:
        print(f"MISSING: {src}")

OK: /kaggle/working/rq2_frameworkA_ragas_cheap_checkpoint.jsonl  (685 bytes)
OK: /kaggle/working/rq2_frameworkA_ragas_cp_checkpoint.jsonl  (2257 bytes)


In [14]:
# Quick QA: which query got the lowest faithfulness, and does that look right?
import json

scored = []
with open("/kaggle/working/rq2_frameworkA_ragas_cheap_checkpoint.jsonl") as f:
    for line in f:
        line = line.strip()
        if line:
            scored.append(json.loads(line))

low_f = sorted(scored, key=lambda r: r["faithfulness"])[:3]
print("=== 3 lowest faithfulness queries ===")
for r in low_f:
    qid = r["query_id"]
    src = next(x for x in eval_rows if x["query_id"] == qid)
    print(f"\n{qid}  faithfulness={r['faithfulness']:.3f}  relevancy={r['answer_relevancy']:.3f}")
    print(f"  Q: {src['user_input']}")
    print(f"  A (first 400 chars): {src['response'][:400]}")

=== 3 lowest faithfulness queries ===

q028  faithfulness=0.650  relevancy=0.990
  Q: How do metabolic flux models identify bottlenecks in cellular energy production?
  A (first 400 chars): Metabolic flux models identify bottlenecks and characterize cellular energy production by employing constraint-based approaches that integrate stoichiometric, thermodynamic, and kinetic data to map reaction flux configurations [1][5][8]. By applying optimization principles, such as the maximization of biomass production or the minimization of flux costs, these models reduce the solution space to i

q059  faithfulness=0.727  relevancy=0.797
  Q: How does frequency-dependent selection maintain genetic polymorphism in natural populations?
  A (first 400 chars): Frequency-dependent selection maintains genetic polymorphism in natural populations through rapid temporal fluctuations in environmental conditions, such as seasonal climate changes [4]. These fluctuations impose variable selection pressures tha

## Cell 6 - Aggregate: mean + bootstrap 95% CI per metric

Same convention as Framework A's amplification result (`rq2_frameworkA_summary.md`):
query-level bootstrap, seed 42, 10,000 resamples, reported alongside the point
estimate rather than a bare mean.


In [13]:
# Cell 6: aggregate CHEAP metric scores with bootstrap 95% CI
import numpy as np
import json

CKPT_PATH = "/kaggle/working/rq2_frameworkA_ragas_cheap_checkpoint.jsonl"

scored = []
with open(CKPT_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            scored.append(json.loads(line))
print(f"loaded {len(scored)} scored queries")

RNG_SEED = 42
N_BOOT = 10_000
rng = np.random.default_rng(RNG_SEED)

ragas_summary = {}
for name in METRICS_CHEAP:
    vals = np.array(
        [r[name] for r in scored if r.get(name) is not None and not np.isnan(r[name])],
        dtype=float,
    )
    n = len(vals)
    if n == 0:
        ragas_summary[name] = {"mean": None, "n_queries": 0, "bootstrap_95CI": [None, None]}
        continue
    point = float(np.mean(vals))
    boot_means = [float(np.mean(rng.choice(vals, size=n, replace=True))) for _ in range(N_BOOT)]
    ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
    ragas_summary[name] = {
        "mean": round(point, 4),
        "n_queries": n,
        "n_dropped_null": len(scored) - n,
        "bootstrap_95CI": [round(float(ci_low), 4), round(float(ci_high), 4)],
        "min": round(float(np.min(vals)), 4),
        "max": round(float(np.max(vals)), 4),
    }

print("\n=== RAGAS cheap-metric summary (Framework A, n=100) ===")
print(json.dumps(ragas_summary, indent=2))

loaded 100 scored queries

=== RAGAS cheap-metric summary (Framework A, n=100) ===
{
  "faithfulness": {
    "mean": 0.9818,
    "n_queries": 100,
    "n_dropped_null": 0,
    "bootstrap_95CI": [
      0.9706,
      0.9911
    ],
    "min": 0.65,
    "max": 1.0
  },
  "answer_relevancy": {
    "mean": 0.9179,
    "n_queries": 100,
    "n_dropped_null": 0,
    "bootstrap_95CI": [
      0.9048,
      0.9301
    ],
    "min": 0.6992,
    "max": 1.0
  }
}


## Cell 7 - Consolidate with existing NDCG@10 / MRR / SPD, save final files

These three numbers are copied from the already-measured, already-committed
results (not recomputed here):
- `NDCG@10` / `MRR` : `results/rq3_results.json` -> `baseline` (lambda=1.0,
  i.e. pre-re-ranking Step 5a retrieval; Step 6/RQ3 numbers are re-ranking
  variants on top of this baseline)
- `SPD` : `results/rq1_optionB_result.json` (RQ1, Option B, QS Top-50)

If RQ1 or RQ3 gets re-run with different numbers, update the constants below
to match - this cell does not read those files live because they are not
attached as a Kaggle input for this notebook (they are tiny and already
final, so we copy the values with the source noted).


In [ ]:
# Cell 7: consolidate everything into one file for Step 9
EXISTING_METRICS = {
    "NDCG@10": {"value": 0.8092, "source": "results/rq3_results.json (baseline, lambda=1.0)"},
    "MRR": {"value": 0.7506, "source": "results/rq3_results.json (baseline, lambda=1.0)"},
    "SPD": {
        "point": 0.029,
        "bootstrap_95CI": [-0.0054, 0.0648],
        "significant": False,
        "source": "results/rq1_optionB_result.json (RQ1, Option B, QS Top-50)",
    },
}

consolidated = {
    "step": 8,
    "owner": "Raj",
    "judge_model": JUDGE_MODEL,
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "ragas": ragas_summary,
    "existing_metrics": EXISTING_METRICS,
    "n_queries_scored": len(scored),
}

RESULT_PATH = "/kaggle/working/rq2_frameworkA_ragas_results.json"
with open(RESULT_PATH, "w") as f:
    json.dump(ragas_summary, f, indent=2)
print("saved ->", RESULT_PATH)

CONSOLIDATED_PATH = "/kaggle/working/step8_consolidated_summary.json"
with open(CONSOLIDATED_PATH, "w") as f:
    json.dump(consolidated, f, indent=2)
print("saved ->", CONSOLIDATED_PATH)

print("\n--- Step 8 consolidated summary ---")
print(json.dumps(consolidated, indent=2))


## Cell 8 - Step 8 conclusion (fill in after the run completes)

**Pre-registered scope.** RAGAS (Faithfulness, Answer Relevancy, Context
Precision) evaluates whether the RQ2 Framework A answers are actually GOOD,
independent of the fairness question RQ1-RQ3 already answered. This is not a
bias measurement; it is the quality stamp described in the RAGAS overview
shared with the team.

**Result.** *(fill in mean +/- 95% CI for each of the three RAGAS metrics once
the batch run finishes, plus any queries that failed after MAX_RETRIES)*

**How this fits with RQ1/RQ2/RQ3.** RQ1 (retrieval), RQ2 Framework A
(generation-stage citation amplification), and RQ3 (re-ranking) all measured
weak/non-significant institutional bias. RAGAS closes the separate question of
whether the system's answers are faithful, relevant, and well-grounded -
letting the final report say the system is *both* fair *and* good, rather
than just fair.

**Honest limits.** Reference-free RAGAS scores are themselves LLM-as-judge
estimates from the same class of model used for generation (gemini-3.1-flash-lite);
they are not independent of judge quality/bias, and should be read as
directional evidence, not ground truth.

**Next.** Step 9 (Streamlit dashboard) reads `step8_consolidated_summary.json`
for the numbers under Yan-Bo's "metrics at the top" layout, alongside
Jici's `step9_streamlit_demo_draft.py` UI skeleton.
